# <font color="#F48E16">XAI aplicada al Deep Learning: Ejercicio práctico: XAI genérico con MNIST</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 12 de mayo de 2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from time import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PATH = "/content/drive/MyDrive/CURSO XAI/BLOQUE 2 - Explicabilidad/"

## <font color="#F48E16">Descarga del dataset y preprocesado</font>

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
assert x_train.shape == (60000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert y_train.shape == (60000,)
assert y_test.shape == (10000,)

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

mean = x_train.mean(axis=0)
x_train = x_train - mean[None, :, :]
x_test = x_test - mean[None, :, :]

In [ ]:
vmax = np.abs(x_train[0]).max()
plt.imshow(x_train[0], vmin=-vmax, vmax=vmax, cmap="bwr")
plt.colorbar()
plt.show()

## <font color="#F48E16">Distintos algoritmos de XAI</font>

In [ ]:
def importancia_permutacion(X, y, model, N=10):
    base_score = model.evaluate(X, y, verbose=False)[-1]
    num_atributos = X.shape[1] * X.shape[2]
    Rx = np.zeros(num_atributos)

    for i in range(num_atributos):
      for j in range(N):
        X_perm = X.copy().reshape(-1, num_atributos)
        np.random.shuffle(X_perm[:, i])
        Rx[i] += model.evaluate(X_perm.reshape(-1, X.shape[1], X.shape[2]), y, verbose=False)[-1]

    Rx = base_score - Rx / N

    return Rx

In [ ]:
def relevancia_oclusion(X, y, model, verbose=True):
  if X.ndim == 2: # 1 solo dato, lo ponemos en formato matricial
    X = X[None, :, :]
    pred = model.predict(X, verbose=False)
  else:
    score = model.evaluate(X, y, verbose=False)[-1]

  num_atributos = X.shape[1] * X.shape[2]

  Rx = np.zeros(num_atributos)
  for i in range(num_atributos):
    X_null = X.copy().reshape(-1, num_atributos)
    X_null[:, i] = 0
    X_null = X_null.reshape(-1, X.shape[1], X.shape[2])

    if X_null.shape[0] == 1: # Soy local
      Rx[i] = pred[0, y] - model.predict(X_null, verbose=False)[0,y]
    else: # Soy global
      Rx[i] = score - model.evaluate(X_null, y, verbose=False)[-1]

  return Rx

In [ ]:
def shap(X, y, model, item, N=10, verbose=True):
    num_atributos = X.shape[1] * X.shape[2]
    X_reshaped = X.reshape(-1, num_atributos)
    shap_values = np.zeros(num_atributos)
    x = X_reshaped[item][None, :]
    y = y[item]

    baseline = model.predict(X, verbose=False).mean(axis=0, keepdims=True)

    iterador_N = tqdm(range(N)) if verbose else range(N)
    for _ in iterador_N:
      prev_pred = baseline
      perm = np.random.permutation(num_atributos)
      for c in range(0, num_atributos, 8): # Voy cambiando de 8 en 8 para ir más rápido
        x_S = x.copy()
        x_S[:, perm[c+1:]] = X_reshaped.mean(axis=0)[perm[c+1:]] # Esta estrategia puede ser diferente
        pred = model.predict(x_S.reshape(-1, X.shape[1], X.shape[2]), verbose=False)
        shap_values[perm[c-8:c]] += (pred - prev_pred)[0,y]
        prev_pred = pred

    return shap_values / N

In [ ]:
def lime(X, y, model, item, D=1000):
  num_atributos = X.shape[1] * X.shape[2]
  X_reshaped = X.reshape(-1, num_atributos)
  x0 = X_reshaped[item][None, :]
  y = y[item]

  # 1. Generamos dataset sintetico X_perturbed alrededor de x0
  X_perturbed = np.repeat(x0, D, axis=0)
  noise = np.random.normal(size=X_perturbed.shape) * X_reshaped.std(axis=0, keepdims=True)
  X_perturbed += noise

  # 2. Se calculan las predicciones del modelo sobre X_perturbed
  y_perturbed = model.predict(X_perturbed.reshape(-1, X.shape[1], X.shape[2]), verbose=False)[:, y]

  # 3. Se asignan pesos a los puntos de X_perturbed según la distancia a x0
  distances = np.linalg.norm(X_perturbed - x0, axis=1) + 1e-8
  weights = 1 / distances # más distancia, menor peso. Esta estrategia puede ser diferente

  # 4. Se entrena un modelo lineal con los puntos de X_perturbed
  linear_model = LinearRegression()
  linear_model.fit(X_perturbed, y_perturbed, sample_weight=weights) # NO OLVIDAR sample_weights
  Rx = linear_model.coef_

  return Rx

## <font color="#F48E16">Modelo</font>

In [ ]:
def build_function(units, optimizer="Adam", activation="tanh"):
    input_layer = tf.keras.layers.Input(shape=(28, 28))
    flatten_layer = tf.keras.layers.Flatten()(input_layer)
    dense_layer = tf.keras.layers.Dense(units, activation=activation)(flatten_layer)
    output_layer = tf.keras.layers.Dense(10, activation="softmax")(dense_layer)

    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

    return model

## <font color="#F48E16">Entrenamiento</font>

In [ ]:
units = 20
learning_rate = 1e-3
activation = "relu"
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model = build_function(units, optimizer, activation)
model.summary()

In [ ]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), batch_size=32, epochs=10)

In [ ]:
model.save_weights(PATH + "models/model__"+activation+".weights.h5")

## <font color="#F48E16">Anular vs Dirección de cambio</font>

In [ ]:
x_train_small = x_train[:100]
y_train_small = y_train[:100]

In [ ]:
time_ini = time()
R_permutacion = importancia_permutacion(x_train_small, y_train_small, model, N=1) # OJO 1 sola permutacion por atributo... 784 atributos
time_fin = time()
print(" > Tiempo de ejecución de Permutación:", time_fin - time_ini)

In [ ]:
time_ini = time()
R_oclusion = relevancia_oclusion(x_train_small, y_train_small, model) # OJO 784 atributos
time_fin = time()
print(" > Tiempo de ejecución de Oclusión:", time_fin - time_ini)

In [ ]:
time_ini = time()
R_shap = [shap(x_train_small, y_train_small, model, item=i, N=1, verbose=False) for i in tqdm(range(len(x_train_small)))]
R_shap = np.array(R_shap)
# OJO recuerda que vamos de 8 en 8 atributos para acelerar
# OJO ten en cuenta que hay que hacer un shap por cada imagen del dataset
time_fin = time()
print(" > Tiempo de ejecución de SHAP:", time_fin - time_ini)

In [ ]:
time_ini = time()
R_lime = [lime(x_train_small, y_train_small, model, item=i, D=1000) for i in tqdm(range(len(x_train_small)))]
R_lime = np.array(R_lime)
# OJO ten en cuenta que hay que hacer un lime por cada imagen del dataset
time_fin = time()
print(" > Tiempo de ejecución de LIME:", time_fin - time_ini)

In [ ]:
vmax = np.abs(R_permutacion.max())
plt.figure(figsize=(4,4))
plt.imshow(R_permutacion.reshape(28,28), vmin=-vmax, vmax=vmax, cmap="bwr")
plt.colorbar()
plt.title("Permutación")
plt.show()

In [ ]:
vmax = np.abs(R_oclusion.max())
plt.figure(figsize=(4,4))
plt.imshow(R_oclusion.reshape(28,28), vmin=-vmax, vmax=vmax, cmap="bwr")
plt.colorbar()
plt.title("Oclusión")
plt.show()

In [ ]:
item = 0
vmax = np.abs(R_lime[item].max())
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(x_train_small[item])
plt.title("Imagen")
plt.subplot(1,2,2)
plt.imshow(R_lime[item].reshape(28,28), vmin=-vmax, vmax=vmax, cmap="bwr")
plt.colorbar()
plt.title("LIME")
plt.show()

In [ ]:
R_limexinput = R_lime * x_train_small.reshape(-1, 28*28)
R_limexinput.shape

In [ ]:
item = 0
vmax = np.abs(R_limexinput[item].max())
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(x_train_small[item])
plt.title("Imagen")
plt.subplot(1,2,2)
plt.imshow(R_limexinput[item].reshape(28,28), vmin=-vmax, vmax=vmax, cmap="bwr")
plt.colorbar()
plt.title("LIME x Input")
plt.show()

## <font color="#F48E16">Curvas insertion-deletion</font>

In [ ]:
# Rx tiene que entrar en formato vector, no matriz
def prueba1(X, y, model, Rx, local=False, verbose=True):
  at = Rx.shape[-1]
  iterador_at = tqdm(range(0, at, 8)) if verbose else range(0, at, 8) # Voy de 8 en 8 para acelerar
  m_all = []
  if local:
    for i in iterador_at:
      ar = at - i
      X_null = np.copy(X)
      for j, (x, r) in enumerate(zip(X, Rx)):
        umbral = np.sort(r)[i]
        X_null[j, r.reshape(28, 28) <= umbral] = 0 # Asignamos la media
      m = model.evaluate(X_null, y, verbose=False)[-1]
      m_all.append(m)
  else:
    for i in iterador_at:
      ar = at - i
      umbral = np.sort(Rx)[i]
      X_null = np.copy(X)
      X_null[:, Rx.reshape(28, 28) <= umbral] = 0 # Asignamos la media
      m = model.evaluate(X_null, y, verbose=False)[-1]
      m_all.append(m)

  return np.array(m_all)

In [ ]:
# Rx tiene que entrar en formato vector, no matriz
def prueba2(X, y, model, Rx, local=False, verbose=True):
  at = Rx.shape[-1]
  iterador_at = tqdm(range(0, at, 8)) if verbose else range(0, at, 8) # Voy de 8 en 8 para acelerar
  m_all = []
  if local:
    for i in iterador_at:
      ar = at - i
      X_null = np.copy(X)
      for j, (x, r) in enumerate(zip(X, Rx)):
        umbral = np.sort(r)[i]
        X_null[j, r.reshape(28, 28) >= umbral] = 0 # Asignamos la media
      m = model.evaluate(X_null, y, verbose=False)[-1]
      m_all.append(m)
  else:
    for i in iterador_at:
      ar = at - i
      umbral = np.sort(Rx)[i]
      X_null = np.copy(X)
      X_null[:, Rx.reshape(28, 28) >= umbral] = 0 # Asignamos la media
      m = model.evaluate(X_null, y, verbose=False)[-1]
      m_all.append(m)

  return np.array(m_all)

In [ ]:
prueba1_permutacion = prueba1(x_train_small, y_train_small, model, R_permutacion, local=False)

In [ ]:
prueba2_permutacion = prueba2(x_train_small, y_train_small, model, R_permutacion, local=False)

In [ ]:
plt.plot(prueba1_permutacion, label="Prueba 1")
plt.plot(prueba2_permutacion, label="Prueba 2")
plt.title("Permutación")
plt.legend()
plt.show()

In [ ]:
prueba1_oclusion = prueba1(x_train_small, y_train_small, model, R_oclusion, local=False)

In [ ]:
prueba2_oclusion = prueba2(x_train_small, y_train_small, model, R_oclusion, local=False)

In [ ]:
plt.plot(prueba1_oclusion, label="Prueba 1")
plt.plot(prueba2_oclusion, label="Prueba 2")
plt.title("Oclusión")
plt.show()

In [ ]:
prueba1_limexinput = prueba1(x_train_small, y_train_small, model, R_limexinput.reshape(-1, 28*28), local=True)

In [ ]:
prueba2_limexinput = prueba2(x_train_small, y_train_small, model, R_limexinput.reshape(-1, 28*28), local=True)

In [ ]:
plt.plot(prueba1_limexinput, label="Prueba 1")
plt.plot(prueba2_limexinput, label="Prueba 2")
plt.title("LIME x Input")
plt.show()

## <font color="#F48E16">Coeficiente de Explicabilidad-Rendimiento</font>

In [ ]:
at = 28*28
iterador_at = np.arange(0, at, 8) # Voy de 8 en 8 para definir los saltos para el EPC
ar = at - iterador_at
factor1 = (at - ar) / at
m = model.evaluate(x_train_small, y_train_small, verbose=False)[-1]

In [ ]:
epc_permutacion = factor1 * (prueba1_permutacion - prueba2_permutacion) / m

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.plot(prueba1_permutacion, label="Prueba 1")
plt.plot(prueba2_permutacion, label="Prueba 2")
plt.title("Permutación")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_permutacion, label="EPC")
plt.legend()
plt.title("Permutación")
plt.show()

In [ ]:
epc_oclusion = factor1 * (prueba1_oclusion - prueba2_oclusion) / m

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.plot(prueba1_oclusion, label="Prueba 1")
plt.plot(prueba2_oclusion, label="Prueba 2")
plt.title("Oclusión")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_oclusion, label="EPC")
plt.legend()
plt.title("Oclusión")
plt.show()

In [ ]:
epc_limexinput = factor1 * (prueba1_limexinput - prueba2_limexinput) / m

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.plot(prueba1_limexinput, label="Prueba 1")
plt.plot(prueba2_limexinput, label="Prueba 2")
plt.title("LIME x Input")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_limexinput, label="EPC")
plt.legend()
plt.title("LIME x Input")
plt.show()

In [ ]:
imax = np.argmax(epc_limexinput)
print(imax)

plt.figure(figsize=(10, 3))
plt.plot(epc_permutacion, label="Permutación")
plt.plot(epc_oclusion, label="Oclusión")
plt.plot(epc_limexinput, label="LIME x Input")
plt.title("EPC")
plt.legend()
plt.show()